In [ ]:
pc = 'FotR_ex'
import os
config = {
    'root_dir':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/',
        'cluster': '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_extended/outputs/',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/'
    },
    'folder_to_save':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/GMM_TIFON/',
        'cluster': '/home/m.jaraiz/Documentos/GMM/GMM_TIFON/',
        'FotR_ex': './example_GMM/'
    },
    'pylom_path': {
        'laptop': '/home/migueljaraiz/anaconda3/repos/pyLowOrder',
        'cluster': '/home/m.jaraiz/repos/pyLowOrder',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/pyLowOrder'
    }   
}

for key in ['root_dir', 'pylom_path']:
    config[key]['FotR_ex'] = [path for _, path in config[key].items()]


import sys
sys.stdout = open(sys.stdout.fileno(), mode='w', buffering=1)

try:
    import pyLOM
    print('Entorno con pyLOM instalado')
except ImportError as e:
    print(e)
    print('Imported by local repository')

    if pc == 'FotR_ex':
        for path in config['pylom_path'][pc]:
            if os.path.exists(path):
                sys.path.append(path)
                print(f'Found pylom path: {path}')
                break
        print('Not found pylom path')
    else:
        sys.path.append(config['pylom_path'][pc])

from FotR import FRODO
import matplotlib.pyplot as plt

import numpy as np, torch

def symlog(x, linthresh=1e-3):
    if isinstance(x, torch.Tensor):
        return torch.sign(x) * torch.log10(1 + torch.abs(x) / linthresh)
    elif isinstance(x, np.ndarray):
        return np.sign(x) * np.log1p(np.abs(x) / linthresh)

for folder_path in config['root_dir'][pc]:
    if os.path.exists(folder_path):
        config['root_dir'][pc] = folder_path
        print('Folder root_dir found succesfull.')
        break
if isinstance(config['root_dir'][pc], list):
    raise LookupError('Folder root_dir not found with this configuration.')
    
db = FRODO(
    root_dir = config['root_dir'][pc],
    format = 'PYLOM',
    file = 'CADGroup_3_completo_stage_1.h5')

db.extract_inputs(
    keys_inputs={
        'ptos': 'xyz',    # coordenadas del mallado → data_dict['inputs']['ptos']
        'aoa': 'aoa',   # variable paramétrica   → data_dict['inputs']['aoa']
        'mach': 'M',   # variable paramétrica   → data_dict['inputs']['M']
    },
    keys_aux={},
)

db.extract_outputs(
    keys_outputs={
        'cp': 'BoundaryValues_CoefPressure',
        # 'gradrho': 'AugStateGrad_DensityGradient_interp',  # field del Dataset → data_dict['outputs']['cp']
        # 'gradT': 'AugStateGrad_TemperatureGradient_interp',
        # 'T': 'AugState_Temperature_interp',
        # 'rho': 'State_Density_interp'
        }
    
)

In [ ]:
# Acceder a los datos
xyz = db.sets.get_xyz()           # (npoints, 3)
aoa   = db.sets.get_variable('aoa') # (500,)
mach = db.sets.get_variable('mach')   # (500,)
cp  = db.sets.get_field('cp')     # (npoints, 500)

# T = db.sets.get_field('T')
# rho = db.sets.get_field('rho')

# gradrhox = db.sets.get_field('gradrho')[0, :, :]#np.linalg.norm(db.sets.get_field('gradrho'), ord=2,axis=0)
# gradTx = db.sets.get_field('gradT')[0, :, :]#np.linalg.norm(db.sets.get_field('gradT'), ord=2,axis=0)


from FotR import SAM
xyz_sort, order_sort = SAM.Weapons.sort_by_centroid(xyz)
cp_sort = cp[order_sort, :]

# T_sort = T[order_sort, :]
# rho_sort = rho[order_sort, :]

# gradrhox_sort = gradrhox[order_sort, :]
# gradTx_sort = gradTx[order_sort, :]

sep = 1
n_clusters = 2
def segmentar_tensor(tensor, sep):
    return torch.from_numpy(tensor[::sep, :]) if sep != 1 else torch.from_numpy(tensor)

tensor_ptos = segmentar_tensor(xyz_sort, sep)
tensor_cp = segmentar_tensor(cp_sort, sep)

# tensor_rho = segmentar_tensor(rho_sort, sep)
# tensor_T = segmentar_tensor(T, sep)

# tensor_gradrhox = segmentar_tensor(gradrhox_sort, sep)
# tensor_gradTx = segmentar_tensor(gradTx_sort, sep)

from scipy.signal import savgol_filter

activate_filter = False

window_length = 101   # debe ser impar
polyorder = 2

def SGS(tensor, window_length=21, polyorder=3):

    def _filter(x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()

            if x_np.ndim == 1:
                y = savgol_filter(
                    x_np,
                    window_length=window_length,
                    polyorder=polyorder
                )
            else:
                y = savgol_filter(
                    x_np,
                    window_length=window_length,
                    polyorder=polyorder,
                    axis=0
                )

            return torch.from_numpy(y).to(x.dtype)

        elif isinstance(x, np.ndarray):

            if x.ndim == 1:
                return savgol_filter(
                    x,
                    window_length=window_length,
                    polyorder=polyorder
                )
            else:
                return savgol_filter(
                    x,
                    window_length=window_length,
                    polyorder=polyorder,
                    axis=0
                )

        else:
            raise ValueError(
                "Input must be a torch tensor or numpy array."
            )

    if isinstance(tensor, (list, tuple)):
        return [_filter(t) for t in tensor]
    else:
        return _filter(tensor)

if activate_filter:
    (
        tensor_cp_filtered,
        # tensor_gradrhox_filtered,
        # tensor_gradTx_filtered
        # tensor_rho_filtered,
        # tensor_T_filtered
    ) = SGS(
        [
            tensor_cp,
            # tensor_T,
            # tensor_rho,
            # tensor_gradrhox,
            # tensor_gradTx
            ],
        window_length=21,
        polyorder=3
    )
else:
    (
        tensor_cp_filtered,
        # tensor_rho_filtered,
        # tensor_T_filtered,
        
        # tensor_gradrhox_filtered,
        # tensor_gradTx_filtered
    ) = (
        tensor_cp,
        # tensor_T,
        # tensor_rho,
        
        # tensor_gradrhox,
        # tensor_gradTx
    )
    
scale_log = True

In [ ]:
stencil = 200
dcp_ds = torch.zeros(tensor_cp_filtered.shape, dtype=torch.float64)
dcp2_ds = torch.zeros(tensor_cp_filtered.shape, dtype=torch.float64)

# drho_ds = torch.zeros(tensor_rho_filtered.shape, dtype=torch.float64)
# drho2_ds = torch.zeros(tensor_rho_filtered.shape, dtype=torch.float64)

# dT_ds = torch.zeros(tensor_T_filtered.shape, dtype=torch.float64)
# dT2_ds = torch.zeros(tensor_T_filtered.shape, dtype=torch.float64)

for case in range(tensor_cp_filtered.shape[1]):
    dcp_ds[:, case] = SAM.Weapons.surface_derivative(
        X=tensor_ptos,
        f=tensor_cp_filtered[:, case],
        order=1,
        stencil_width=stencil,   
        poly_order=polyorder,
    )
    
    # drho_ds[:, case] = SAM.Weapons.surface_derivative(
    #     X=tensor_ptos,
    #     f=tensor_rho_filtered[:, case],
    #     order=1,
    #     stencil_width=stencil,   
    #     poly_order=polyorder,
    # )
    
    # dT_ds[:, case] = SAM.Weapons.surface_derivative(
    #     X=tensor_ptos,
    #     f=tensor_T_filtered[:, case],
    #     order=1,
    #     stencil_width=stencil,   
    #     poly_order=polyorder,
    # )
    
dcp_ds_filtered = SGS(dcp_ds, window_length=5, polyorder=2) if activate_filter else dcp_ds
# drho_ds_filtered = SGS(drho_ds, window_length=5, polyorder=2) if activate_filter else drho_ds
# dT_ds_filtered = SGS(dT_ds, window_length=5, polyorder=2) if activate_filter else dT_ds

for case in range(tensor_cp_filtered.shape[1]):
    dcp2_ds[:, case] = SAM.Weapons.surface_derivative(
        X=tensor_ptos,
        f=dcp_ds_filtered[:, case],
        order=1,
        stencil_width=stencil,
        poly_order=polyorder,
    )

    # drho2_ds[:, case] = SAM.Weapons.surface_derivative(
    #     X=tensor_ptos,
    #     f=drho_ds_filtered[:, case],
    #     order=1,
    #     stencil_width=stencil,   
    #     poly_order=polyorder,
    # )
    
    # dT2_ds[:, case] = SAM.Weapons.surface_derivative(
    #     X=tensor_ptos,
    #     f=dT_ds_filtered[:, case],
    #     order=1,
    #     stencil_width=stencil,   
    #     poly_order=polyorder,
    # )
    
dcp_ds_log = symlog(dcp_ds_filtered, linthresh=1e-4) if scale_log else dcp_ds_filtered
dcp2_ds_log = symlog(dcp2_ds, linthresh=1e-4) if scale_log else dcp2_ds

# drho_ds_log = symlog(drho_ds_filtered, linthresh=1e-4) if scale_log else drho_ds_filtered
# drho2_ds_log = symlog(drho2_ds, linthresh=1e-4) if scale_log else drho2_ds

# dT_ds_log = symlog(dT_ds_filtered, linthresh=1e-4) if scale_log else dT_ds_filtered
# dT2_ds = symlog(dT2_ds, linthresh=1e-4) if scale_log else dT2_ds


# gradrhox_log = symlog(tensor_gradrhox_filtered, linthresh=1e-4) if scale_log else tensor_gradrhox_filtered
# gradTx_log = symlog(tensor_gradTx_filtered, linthresh=1e-4) if scale_log else tensor_gradTx_filtered

db.sets.add_aux(
    array_name = 'dcp_ds_log',
    array = dcp_ds_log.numpy(),
    notes = 'Log dcp_ds')

db.sets.add_aux(
    array_name = 'dcp2_ds_log',
    array = dcp2_ds_log.numpy(),
    notes = 'Log dcp2_ds')

# db.sets.add_aux(
#     array_name = 'drho_ds_log',
#     array = drho_ds_log.numpy(),
#     notes = 'Log drho_ds')

# db.sets.add_aux(
#     array_name = 'drho2_ds_log',
#     array = drho2_ds_log.numpy(),
#     notes = 'Log drho2_ds')

# db.sets.add_aux(
#     array_name = 'dT_ds_log',
#     array = dT_ds_log.numpy(),
#     notes = 'Log dT_ds')

# db.sets.add_aux(
#     array_name = 'dT2_ds',
#     array = dT2_ds.numpy(),
#     notes = 'Log dT2_ds')
# db.sets.add_aux(
#     array_name = 'gradrhox_log',
#     array = gradrhox_log.numpy(),
#     notes = 'Log gradrhox'
# )

# db.sets.add_aux(
#     array_name = 'gradTx_log',
#     array = gradTx_log.numpy(),
#     notes = 'Log gradTx'
# )

db.data_dict['inputs']['ptos'] = tensor_ptos.numpy()
db.data_dict['outputs']['cp'] = tensor_cp_filtered.numpy()
# [db.data_dict['outputs'].pop(key, None) for key in ['gradT', 'gradrho']]
db.sets.create_jset(verbose=False)
# display(db.df_data)


In [ ]:
from FotR import GIMLI
features = ['dcp_ds_log', 'dcp2_ds_log']#, 'drho_ds_log', 'drho2_ds_log', 'dT_ds_log', 'dT2_ds'] # , 'gradrhox_log', 'gradTx_log'

gmm = (
    GIMLI(
        db.df_data,
        variables=features,
        groups=["aoa", "mach"],
        algorithm="gmm",
        scaler="robust",
    )
    .prepare()
    .fit(n_clusters=2)
    .predict()
)